This notebook introduces how to use DeepPlant to predict chromatin state (CPS) and enhancer activity (EAP), for Arabidopsis thaliana (CSP, EAP), Oryza sativa (CSP).

If you use our codes in your research, please cite:

Daoud, Ahmed, et al. "Deep-Plant: a supervised foundation model for plant regulatory genomics." bioRxiv (2026): 2026-04.

## Note: DeepPlant operates on 2.5 kb DNA sequences, with a primary focus on the central region.
## The flanking regions provide contextual information, while predictions are made for the central 200 bp. In particular, the model is designed to capture chromatin state features and identify enhancer activity concentrated in this central window.
## Note: Input sequences are normalized to a fixed length of 2.5 kb.
## If a longer sequence is provided, only the central 2.5 kb window is used.
## If a shorter sequence is provided, it is padded with 'N's to reach 2.5 kb.

In [47]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Load the variables from the .env file
load_dotenv()

# 2. Grab the global path
deep_plant_path = os.getenv("DEEPPLANTPATH")

if not deep_plant_path:
    raise ValueError("DEEPPLANTPATH is missing! Did you create your .env file using 'python3 setup_env.py'?")
else:
    print("DEEPPLANTPATH is",deep_plant_path)


DEEPPLANTPATH is /s/chromatin/m/nobackup/ahmed/GitRepositries/DeepPlant


# Chromatin state prediction For arabidopsis and rice

In [48]:
from src.DeepPlant import build_model
from src.config import CSPConfig
from src.utils import hot_encode_sequence,read_json,get_device,get_DNA_sequence,read_fasta_file,create_path
import torch
import pandas as pd

### You can retrieve a sequence directly from the genome by specifying
### organism, chromosome, and start position.
### After executing the cell below, follow the displayed instructions to explore the full set of result details.

In [ ]:
organism = 'Arabidopsis' # Arabidopsis / Rice
chromosome = 4
start = 8900
end = start+2500
DNA_sequence = get_DNA_sequence(organism=organism,chrom=chromosome,start=start,end=end)
device = get_device()
if organism=='Arabidopsis':
    model = build_model(args=CSPConfig(**read_json(os.path.join(deep_plant_path,'config/config_AT_2500.json'))),new_model=False,model_path=os.path.join(deep_plant_path,'models/model_AT_CSP.pt')).to(device)
    targets = open(os.path.join(deep_plant_path,'data/arabidopsis/CSP/CSP_label.txt')).read().splitlines()
else:
    model = build_model(args=CSPConfig(**read_json(os.path.join(deep_plant_path,'config/config_OS_2500.json'))),new_model=False,model_path=os.path.join(deep_plant_path,'models/model_OS_CSP.pt')).to(device)
    targets = open(os.path.join(deep_plant_path,'data/arabidopsis/CSP/CSP_label.txt')).read().splitlines()
outputs = model(torch.from_numpy(hot_encode_sequence(DNA_sequence)).unsqueeze(0).to(device))[0].detach().cpu().numpy()
results_path = os.path.join(deep_plant_path,f"results/CSP_pred/{organism}_CSP_{chromosome}_{start}_{end}.csv")
metadata_path = os.path.join(deep_plant_path,'data/arabidopsis/CSP/AT_CSP_Metadata.csv')
create_path(os.path.dirname(results_path))
pd.DataFrame({'Target':targets,'Prediction':outputs}).to_csv(results_path,index=False)
print(f"Refer to {results_path} to see the model predictions \n and {metadata_path} to see the full metadata")

Loading model state
Model checkpoint loaded
Refer to /s/chromatin/m/nobackup/ahmed/GitRepositries/DeepPlant/results/CSP_pred/Arabidopsis_CSP_4_8900_11400.csv to see the model predictions 
 and /s/chromatin/m/nobackup/ahmed/GitRepositries/DeepPlant/data/arabidopsis/CSP/AT_CSP_Metadata.csv to see the full metadata


# Enhancer prediction for Arabidopsis

### You can retrieve a sequence directly from the Arabidopsis genome by specifying
### organism, chromosome, and start position.
### After executing the cell below, the model will output the probability of an enhancer being present within the provided DNA sequence.

In [33]:
from src.DeepPlant_enhancer import build_model
from src.config import EnhancerConfig
from src.utils import hot_encode_sequence,read_json,get_device,get_DNA_sequence
import torch

In [ ]:
organism = 'Arabidopsis' # Arabidopsis / rice
chromosome = 4
start = 8900
end = start+2500
DNA_sequence = get_DNA_sequence(organism=organism,chrom=chromosome,start=start,end=end)
device = get_device()
act_fn = torch.nn.Softmax(dim=0)
model = build_model(args=EnhancerConfig(**read_json(os.path.join(deep_plant_path,'config/config_AT_enhancer.json'))),new_model=False,model_path=os.path.join(deep_plant_path,'models/model_AT_EAP.pt')).to(device)
outputs = act_fn(model(torch.from_numpy(hot_encode_sequence(DNA_sequence)).unsqueeze(0).to(device))).detach().cpu().numpy()
print("The probability of an enhancer in the center 400-600 bps of the DNA sequence between %i and %i in chromosome %s is %.4f"%(start,end,chromosome,outputs[1]))

Loading model state
Model checkpoint loaded
The probability of an enhancer in the center 400-600 bps of the DNA sequence between 8900 and 11400 in chromosome 4 is 0.0000
